
# Stack Overflow Developer Survey 2014 - canonical (not yet encoded) version

**Purpose:** one year of a shared multi-year dataset. This notebook produces `canonical_2014.csv`: shared column names, original text answers kept as they are for single-answer questions, multi-answer questions stored as `;`-separated lists (the format 2016+ files use), and the language write-ins cleaned into canonical names with a count. **No one-hot encoding, no imputation, no scaling, no rare-level merging** - those are learned from the data and belong in the pooled pipeline, fitted on the training years only.

**Same rules as the 2011 notebook**, plus the new things this year's file needs (listed in Gate 2). Nothing is assumed until printed; row counts are logged at every drop; one action per cell; every mapping is shown as a table.

**Gate 1 is the same load and normalize steps as before** (including `keep_default_na=False`, so real answers such as `None` survive).


## Gate 1 - Load and normalize

### Cell 1.0 - Imports, file path, decision log
**Purpose:** set up the environment and find the file. The path tries `data/2014/results.csv` first, then `2014_results.csv` in the working directory.
**Decision made:** none. `decision_log` collects every transformation so the final log (Gate 7) is built from what actually ran.

In [1]:
import csv
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 200)

CANDIDATES = [Path("data/2014/results.csv"), Path("2014_results.csv")]
PATH = next((p for p in CANDIDATES if p.exists()), None)
if PATH is None:
    raise FileNotFoundError("none of the candidate paths exist: " + ", ".join(str(p) for p in CANDIDATES))
print("using file:", PATH, "| size in bytes:", PATH.stat().st_size)
print("pandas version:", pd.__version__)

decision_log = []

def log(step, text):
    decision_log.append((step, text))
    print("[LOG]", step, "-", text)

using file: data\2014\results.csv | size in bytes: 6128696
pandas version: 3.1.0


### Cell 1.1 - Detect the file encoding
**Purpose:** `read_csv` defaults to UTF-8 and would fail on this file, so check before loading.
**Decision made:** use UTF-8 if the whole file decodes, otherwise `cp1252`, and print the byte that forced the fallback.

In [2]:
raw_bytes = PATH.read_bytes()
non_ascii = sum(1 for b in raw_bytes if b > 127)
print("total bytes:", len(raw_bytes), "| non-ASCII bytes:", non_ascii)
try:
    raw_bytes.decode("utf-8")
    ENCODING = "utf-8"
    print("utf-8 decodes the whole file cleanly")
except UnicodeDecodeError as err:
    ENCODING = "cp1252"
    print("utf-8 fails at byte", err.start, "- surrounding bytes:", raw_bytes[max(0, err.start - 25):err.start + 10])
    print("falling back to cp1252")
log("1.1 encoding", "file read with encoding=" + ENCODING + " (" + str(non_ascii) + " non-ASCII byte(s) in the file)")

total bytes: 6128696 | non-ASCII bytes: 114
utf-8 decodes the whole file cleanly
[LOG] 1.1 encoding - file read with encoding=utf-8 (114 non-ASCII byte(s) in the file)


### Cell 1.2 - Load with `header=0`, without pandas' automatic NA parsing
**Purpose:** load the CSV exactly as written.
**Decision made:** read every cell as text (`dtype=str`) and set `keep_default_na=False, na_values=[]`. By default pandas converts the literal strings `None`, `N/A`, `NA` and `null` to NaN. In this survey `None` is a real answer (for example the "None" language checkbox), so default parsing would silently erase it. Genuine blanks are converted in Cell 1.4 instead, as your gotcha list prescribes.

In [3]:
df = pd.read_csv(PATH, header=0, encoding=ENCODING, dtype=str, keep_default_na=False, na_values=[])
print("shape after load (rows, columns):", df.shape)

header_looks_like_ints = all(str(c).isdigit() for c in df.columns)
print("column names are all integers (header consumed as data)?", header_looks_like_ints)
if header_looks_like_ints:
    raise ValueError("header row was consumed as data - reload")
print("first 3 column names:", list(df.columns[:3]))
log("1.2 load", "loaded " + str(df.shape[0]) + " rows x " + str(df.shape[1]) + " columns, all as text, default NA parsing off")

shape after load (rows, columns): (7644, 120)
column names are all integers (header consumed as data)? False
first 3 column names: ['What Country do you live in?', 'Unnamed: 1', 'Which US State or Territory do you live in?']
[LOG] 1.2 load - loaded 7644 rows x 120 columns, all as text, default NA parsing off


### Cell 1.3 - Cross-check the load with Python's `csv` module
**Purpose:** confirm pandas did not drop, split or merge rows, and find which header cells are truly empty (pandas renames those `Unnamed: n`, so this is the reliable list).
**Decision made:** none. This only produces `blank_header_positions`, used later to map multi-select spans.

In [4]:
with open(PATH, encoding=ENCODING, newline="") as fh:
    csv_rows = list(csv.reader(fh))
print("rows seen by csv module (header + data):", len(csv_rows))
print("distinct row lengths:", sorted(set(len(r) for r in csv_rows)))
assert len(csv_rows) - 1 == len(df), "pandas and csv module disagree on row count"
assert all(len(r) == df.shape[1] for r in csv_rows), "ragged rows found"

header_raw = csv_rows[0]
blank_header_positions = [i for i, h in enumerate(header_raw) if h.strip() == ""]
print("header cells that are empty:", len(blank_header_positions), "of", len(header_raw))
print("first 10 empty header positions:", blank_header_positions[:10])

rows seen by csv module (header + data): 7645
distinct row lengths: [120]
header cells that are empty: 83 of 120
first 10 empty header positions: [1, 14, 15, 16, 17, 18, 19, 21, 22, 23]


### Cell 1.4 - Strip whitespace from column names
**Purpose:** remove leading/trailing spaces from headers, as in your gotcha list.
**Decision made:** `df.columns = df.columns.astype(str).str.strip()`. Cell output lists exactly which names changed.

In [5]:
changed = [(i, repr(c)) for i, c in enumerate(df.columns) if c != c.strip()]
print("column names with leading/trailing whitespace:", changed)
df.columns = df.columns.astype(str).str.strip()
print("names still with whitespace after strip:", sum(c != c.strip() for c in df.columns))
print("duplicate column names:", df.columns[df.columns.duplicated()].tolist())
log("1.4 strip", str(len(changed)) + " column name(s) had surrounding whitespace and were stripped")

column names with leading/trailing whitespace: []
names still with whitespace after strip: 0
duplicate column names: []
[LOG] 1.4 strip - 0 column name(s) had surrounding whitespace and were stripped


### Cell 1.5 - Normalize blank cells to NaN
**Purpose:** turn empty and whitespace-only cells into NaN, once.
**Decision made:** `df.replace(r"^\s*$", np.nan, regex=True)`. The cell counts empty and whitespace-only cells first, then checks that the NaN count rose by exactly that amount, so we know the replacement actually happened. Literal text such as `None` or `N/A` is **not** touched here; it is listed so you can see it. These counts still include row 0, which Cell 1.6 separates: the literal-text counts therefore include the option-label row.

In [6]:
empty_exact = int((df == "").sum().sum())
ws_only = int(df.apply(lambda s: s.str.match(r"^\s+$")).sum().sum())
na_before = int(df.isna().sum().sum())
print("cells exactly empty:", empty_exact, "| whitespace-only (non-empty):", ws_only, "| NaN before:", na_before)

df = df.replace(r"^\s*$", np.nan, regex=True)

na_after = int(df.isna().sum().sum())
print("NaN after:", na_after, "| increase:", na_after - na_before)
assert na_after - na_before == empty_exact + ws_only, "NaN count did not move by the expected amount"
print("total cells:", df.size, "| share missing: %.1f%%" % (100 * na_after / df.size))

literals = ["None", "N/A", "NA", "n/a", "null", "NULL", "NaN", "nan"]
found = []
for i, col in enumerate(df.columns):
    n = int(df.iloc[:, i].isin(literals).sum())
    if n:
        found.append((i, col[:50], n))
print("literal NA-like text kept as real values (not converted):")
for item in found:
    print("   col", item[0], "|", item[1], "|", item[2], "cells")
log("1.5 blanks", str(empty_exact + ws_only) + " empty/whitespace cells converted to NaN; " + str(sum(f[2] for f in found)) + " literal 'None'-style cells kept as text")

cells exactly empty: 495389 | whitespace-only (non-empty): 0 | NaN before: 0
NaN after: 495389 | increase: 495389
total cells: 917280 | share missing: 54.0%
literal NA-like text kept as real values (not converted):
   col 20 | In an average week, how do you spend your time at  | 339 cells
   col 21 | Unnamed: 21 | 504 cells
   col 22 | Unnamed: 22 | 305 cells
   col 23 | Unnamed: 23 | 1958 cells
   col 24 | Unnamed: 24 | 844 cells
   col 25 | Unnamed: 25 | 385 cells
   col 26 | Unnamed: 26 | 564 cells
   col 27 | Unnamed: 27 | 4221 cells
   col 28 | Unnamed: 28 | 324 cells
   col 53 | Unnamed: 53 | 1 cells
   col 81 | Unnamed: 81 | 5 cells
   col 111 | What advertisers do you remember seeing on Stack O | 118 cells
[LOG] 1.5 blanks - 495389 empty/whitespace cells converted to NaN; 9568 literal 'None'-style cells kept as text


### Cell 1.6 - Separate the option-label row
**Purpose:** your brief says the file has a second header-like row. Inspect row 0 of the data to see what it is before touching it.
**Decision made (applied because you asked for all gates; say so if you want it reversed):** row 0 is fully populated and holds the option label for each multi-select column (`Influencer`, `Java`, `iPhone`, ...), with the word `Response` for single-select columns. It is not a respondent, so it is moved out of `df` into `option_labels` (nothing is lost) and the move is logged as a row removal. The original index is kept, so index `n` always means data row `n` of the file.

In [7]:
first_row = df.iloc[0]
print("non-empty cells in row 0:", int(first_row.notna().sum()), "of", df.shape[1])
print("cells equal to 'Response':", int((first_row == "Response").sum()))
print("labels in row 0 other than 'Response', in column order:")
for i, v in enumerate(first_row):
    if v != "Response":
        print("   col", i, "->", v)

rows_before = len(df)
option_labels = df.iloc[0].copy()
df = df.iloc[1:].copy()
print("rows before:", rows_before, "| rows after:", len(df))
assert len(df) == rows_before - 1
log("1.6 option row", "row 0 moved to option_labels (rows " + str(rows_before) + " -> " + str(len(df)) + ")")

non-empty cells in row 0: 120 of 120
cells equal to 'Response': 25
labels in row 0 other than 'Response', in column order:
   col 1 -> Other (please specify)
   col 13 -> iPhone
   col 14 -> iPad
   col 15 -> Android Phone
   col 16 -> Android Tablet
   col 17 -> Windows Phone
   col 18 -> Other
   col 19 -> No mobile app
   col 20 -> Developing new features
   col 21 -> Refactoring / code quality
   col 22 -> Fixing bugs
   col 23 -> Technical support
   col 24 -> Meetings
   col 25 -> Learning new skills
   col 26 -> Surfing the Internet
   col 27 -> Looking for a new job
   col 28 -> Using Stack Exchange
   col 29 -> I can recommend or request products
   col 30 -> I influence purchasing decisions, but don't have final approval
   col 31 -> I have a discretionary budget at my disposal
   col 32 -> I can buy anything I want without asking anyone
   col 33 -> No Involvement
   col 34 -> Hardware
   col 35 -> Servers
   col 36 -> Software
   col 37 -> User Equipment: Monitors, PCs, Lap

### Cell 1.7 - Fully empty rows
**Purpose:** count respondents with no answers at all. Nothing is dropped in Gate 1; Gate 2 proposes the criteria.
**Decision made:** none.

In [8]:
filled_per_row = df.notna().sum(axis=1)
fully_empty = int((filled_per_row == 0).sum())
print("data rows:", len(df))
print("fully empty rows:", fully_empty, "| dropped in Gate 1: 0")
print("filled cells per row - lowest values (value : number of rows):")
print(filled_per_row.value_counts().sort_index().head(8).to_string())
only_country = int(((filled_per_row == 1) & df.iloc[:, 0].notna()).sum())
print("rows whose ONLY filled cell is Country (col 0):", only_country, "- candidates for Gate 2, not dropped")
log("1.7 empty rows", str(fully_empty) + " fully empty rows found; 0 dropped; " + str(only_country) + " rows have only Country filled (Gate 2 decision)")

data rows: 7643
fully empty rows: 0 | dropped in Gate 1: 0
filled cells per row - lowest values (value : number of rows):
1     249
2      48
5      14
6      87
7      31
8      55
9     147
10     85
rows whose ONLY filled cell is Country (col 0): 249 - candidates for Gate 2, not dropped
[LOG] 1.7 empty rows - 0 fully empty rows found; 0 dropped; 249 rows have only Country filled (Gate 2 decision)


### Cell 1.8 - Indexed column list
**Purpose:** print every column with its index, header text, the option label from row 0, and how many rows have a value. Columns with an empty header are continuations of the question to their left, so they show which question they belong to.
**Decision made:** none. `col_index` is kept for the later gates.

In [9]:
records = []
parent_idx = None
for i, col in enumerate(df.columns):
    is_blank = i in blank_header_positions
    if not is_blank:
        parent_idx = i
    n_filled = int(df.iloc[:, i].notna().sum())
    records.append({
        "idx": i,
        "header": col[:62] if not is_blank else "(blank - continues col " + str(parent_idx) + ")",
        "row0_label": option_labels.iloc[i],
        "n_filled": n_filled,
        "pct_missing": round(100 * (1 - n_filled / len(df)), 1),
    })
col_index = pd.DataFrame(records)
print(col_index.to_string(index=False))

 idx                                                         header                                                                                                                  row0_label  n_filled  pct_missing
   0                                   What Country do you live in?                                                                                                                    Response      7643          0.0
   1                                      (blank - continues col 0)                                                                                                      Other (please specify)         0        100.0
   2                    Which US State or Territory do you live in?                                                                                                                    Response      2108         72.4
   3                                               How old are you?                                                                         

### Cell 1.9 - Gate 1 summary
**Purpose:** final shape and the log so far.
**Decision made:** none. Gate 2 follows.

In [10]:
print("final shape (rows, columns):", df.shape)
print("original data rows in file:", len(csv_rows) - 1, "| rows in df:", len(df), "| rows held in option_labels: 1")
print()
print("decision log so far:")
for step, text in decision_log:
    print("  ", step, "|", text)

final shape (rows, columns): (7643, 120)
original data rows in file: 7644 | rows in df: 7643 | rows held in option_labels: 1

decision log so far:
   1.1 encoding | file read with encoding=utf-8 (114 non-ASCII byte(s) in the file)
   1.2 load | loaded 7644 rows x 120 columns, all as text, default NA parsing off
   1.4 strip | 0 column name(s) had surrounding whitespace and were stripped
   1.5 blanks | 495389 empty/whitespace cells converted to NaN; 9568 literal 'None'-style cells kept as text
   1.6 option row | row 0 moved to option_labels (rows 7644 -> 7643)
   1.7 empty rows | 0 fully empty rows found; 0 dropped; 249 rows have only Country filled (Gate 2 decision)


## Gate 2 - Freeze names, choices, junk rows

### Cell 2.0 - Column plan and `CONFIG`
**Purpose:** one table (`ITEMS`) says which source column(s) become which output column, in questionnaire order. Every source column is checked against a keyword from its question text, and the cell proves that **no source column is left unused** (nothing is dropped silently).
**Kinds:** `single` = copied as typed; `list` = option columns joined into a `;` list; `freetext` = free-text box kept as typed (marked in the header map so you can drop it before modelling); `lang` = language block (Gate 6); `salary` = band text + band midpoint.
**Decisions made (switchable in `CONFIG`):** the rows dropped are the same as in 2011 (junk rows, no salary, `Student / Unemployed`) **plus `Rather not say`**, a salary answer that did not exist in 2011: it carries no amount, like a blank.

In [11]:
import re
import math
import datetime

RAW = list(df.columns)
PARENT = {}
_p = None
for _i in range(len(RAW)):
    if _i not in blank_header_positions:
        _p = _i
    PARENT[_i] = _p

def label(i):
    return option_labels.iloc[i]

def full_header(i):
    return RAW[i] if i not in blank_header_positions else RAW[PARENT[i]] + " -> " + str(label(i))

# kind, canonical name, status, source column (or (first, last+1)), keyword that must appear, note
ITEMS = [
    ("id", "ResponseId", "proposed", None, None, "row number of the respondent in the 2014 file (data row n = file row n+1)"),
    ("year", "survey_year", "proposed", None, None, "constant 2014"),
    ("single", "Country", "confirmed", 0, "country", "2014 has real country names (96 values); 2011-2013 had regions as well, so not directly comparable"),
    ("single", "USState", "proposed", 2, "us state", "only asked of US respondents"),
    ("single", "Age", "confirmed", 3, "how old", "text bands, kept as typed"),
    ("single", "Gender", "proposed", 4, "gender", "new in 2014; kept as typed (Male / Female / Prefer not to disclose); think about whether it belongs in a salary model"),
    ("single", "YearsCode", "confirmed", 5, "years of it", "Excel date damage repaired to '<2','2-5','6-10','11+'; question says IT/programming experience - check vs YearsCode/YearsCodePro"),
    ("single", "DevType", "confirmed", 6, "occupation", "single answer; value set differs from 2011/2012 (adds DevOps)"),
    ("salary", "CompTotal", "confirmed", 7, "compensation", "salary BAND text, kept as typed"),
    ("single", "Industry", "proposed", 8, "industry", ""),
    ("single", "OrgDevCount", "proposed", 9, "developers are employed", "number of developers at the company; bands repaired; 2014 has no company-size or team-size question"),
    ("single", "RemoteWork", "proposed", 10, "work remotely", "Never / Occasionally / Part-time / Full-time; new in 2014"),
    ("single", "RemoteEnjoyment", "proposed", 11, "enjoy working remotely", "only asked of people who work remotely"),
    ("single", "RemoteLocation", "proposed", 12, "remotely most of the time", "only asked of people who work remotely"),
    ("list", "MobilePlatformsSupported", "proposed", (13, 20), "native mobile app", "';' list; includes 'No mobile app'"),
    ("single", "TimeSpend_NewFeatures", "proposed", 20, "developing new features", "hours per week band, as typed"),
    ("single", "TimeSpend_Refactoring", "proposed", 21, "refactoring", "hours per week band"),
    ("single", "TimeSpend_BugFixing", "proposed", 22, "fixing bugs", "hours per week band"),
    ("single", "TimeSpend_TechSupport", "proposed", 23, "technical support", "hours per week band"),
    ("single", "TimeSpend_Meetings", "proposed", 24, "meetings", "hours per week band"),
    ("single", "TimeSpend_Learning", "proposed", 25, "learning new skills", "hours per week band"),
    ("single", "TimeSpend_WebSurfing", "proposed", 26, "surfing", "hours per week band"),
    ("single", "TimeSpend_LookingForJob", "proposed", 27, "looking for a new job", "hours per week band; new in 2014 (2013 asked about commuting instead)"),
    ("single", "TimeSpend_StackExchange", "proposed", 28, "using stack exchange", "hours per week band; new in 2014"),
    ("list", "PurchaseInvolvement", "proposed", (29, 34), "involvement in purchasing", "';' list; same wording as 2013"),
    ("list", "PurchaseTypes", "proposed", (34, 41), "types of purchases", "';' list; adds 'Recruitment Tools & Services'"),
    ("single", "BudgetOutsideExpenditure", "proposed", 41, "budget for outside", "text bracket, kept as typed; 'Don't know' kept as a value; brackets differ slightly from 2013 ('$1,000 - $5,000', '>$100,001')"),
    ("lang", "LanguageHaveWorkedWith", "confirmed", (42, 54), "languages or technologies", "';' list of canonical names; tick-boxes + parsed write-ins; 2014 wording is 'languages or technologies used significantly'"),
    ("list", "TechExcitedAbout", "proposed", (54, 67), "excited about", "';' list of technologies; check against later 'WantToWorkWith' columns"),
    ("single", "OpSys", "confirmed", 67, "operating system", "text as typed (Ubuntu, Debian, Mint, Fedora are separate answers in 2014)"),
    ("list", "TechProductsOwned", "proposed", (68, 82), "technology products", "';' list; 'Other' means the typed answer in TechProductsOwned_OtherText"),
    ("freetext", "TechProductsOwned_OtherText", "proposed", 81, "other (please specify)", "typed text of the 'Other' product box, raw"),
    ("single", "ChangedJobsLast12m", "proposed", 82, "changed jobs", "Yes/No"),
    ("single", "JobFoundVia", "proposed", 83, "find out about your current job", "'Other (please specify)' means the typed answer in JobFoundVia_OtherText"),
    ("freetext", "JobFoundVia_OtherText", "proposed", 84, "other (please specify)", "typed text, raw"),
    ("single", "OpenToNewJob", "proposed", 85, "looking for a job or open", "Yes/No"),
    ("single", "RecruiterContactFrequency", "proposed", 86, "contacted by recruiters", "three text answers"),
    ("single", "ContactPref_Email", "proposed", 87, "email", "preference scale, as typed"),
    ("single", "ContactPref_LinkedIn", "proposed", 88, "linkedin", "preference scale"),
    ("single", "ContactPref_Phone", "proposed", 89, "phone", "preference scale"),
    ("single", "ContactPref_CareersMessage", "proposed", 90, "careers message", "preference scale"),
    ("single", "ContactPref_Twitter", "proposed", 91, "twitter", "preference scale"),
    ("single", "RecruitEmail_Personalized", "proposed", 92, "personalized", "extent scale, as typed"),
    ("single", "RecruitEmail_MentionsCode", "proposed", 93, "mentions my code", "extent scale"),
    ("single", "RecruitEmail_Salary", "proposed", 94, "salary information", "extent scale"),
    ("single", "RecruitEmail_Team", "proposed", 95, "team i will work on", "extent scale"),
    ("single", "RecruitEmail_Culture", "proposed", 96, "company culture", "extent scale"),
    ("single", "RecruitEmail_Benefits", "proposed", 97, "benefits", "extent scale"),
    ("single", "RecruitEmail_CompanyLink", "proposed", 98, "careers company page", "extent scale"),
    ("single", "JobBoardVisitFrequency", "proposed", 99, "visit job boards", "three text answers"),
    ("single", "CareersAware", "proposed", 100, "aware of stack overflow careers", "Yes / No / 'I got my current job via Careers 2.0!'"),
    ("single", "CareersProfile", "proposed", 101, "careers 2.0 profile", "text as typed"),
    ("single", "Ad_Relevant", "proposed", 102, "ads are relevant", "agreement scale, as typed"),
    ("single", "Ad_Entertaining", "proposed", 103, "entertaining", "agreement scale"),
    ("single", "Ad_Informative", "proposed", 104, "informative", "agreement scale"),
    ("single", "Ad_ClickInteresting", "proposed", 105, "click on ads", "agreement scale"),
    ("single", "Ad_TrialPurchase", "proposed", 106, "taken a trial", "agreement scale"),
    ("single", "Ad_CanIndicateAds", "proposed", 107, "indicate ads", "agreement scale; new in 2014"),
    ("single", "Ad_UseAdBlocker", "proposed", 108, "ad blocker", "agreement scale"),
    ("single", "ApptivateAware", "proposed", 109, "aware of the apptivate", "Yes / No / Don't Remember"),
    ("single", "ApptivateParticipated", "proposed", 110, "participate in the apptivate", "Yes / No / Don't Remember"),
    ("freetext", "AdvertisersRemembered", "proposed", 111, "advertisers", "open-ended answer, raw typed text"),
    ("single", "StackOverflowReputation", "proposed", 112, "reputation", "levels relabelled '0','1+','50+',...,'10000+' (same repair as 2012/2013); 'Don't have an account' -> '0'"),
    ("list", "StackOverflowUsage", "proposed", (113, 119), "use stack overflow", "';' list; 'Other' means the typed answer in StackOverflowUsage_OtherText; adds 'Looking for a job' and 'Build my online reputation'"),
    ("freetext", "StackOverflowUsage_OtherText", "proposed", 118, "other (please specify)", "typed text of the 'Other' usage box, raw"),
    ("single", "StackOverflowSolutionFrequency", "proposed", 119, "find solutions", "how often Stack Overflow solves their problems; five text answers"),
]

covered = set()
for kind, name, status, spec, expect, note in ITEMS:
    if spec is None:
        continue
    first = spec[0] if isinstance(spec, tuple) else spec
    text = (full_header(first) + " " + str(label(first))).lower()
    assert expect.lower() in text, (name, expect, text[:120])
    covered |= set(range(spec[0], spec[1])) if isinstance(spec, tuple) else {spec}
    print("ok  %-30s %-8s cols %-10s | %s" % (name, kind, str(spec), full_header(first)[:60]))
UNUSED = {}
UNUSED = {1: "the 'Other (please specify)' box under Country was never filled in, so it carries no information"}
for i, why in UNUSED.items():
    assert int(df[RAW[i]].notna().sum()) == 0, "column " + str(i) + " has answers, so it cannot be declared unused"
    print("unused on purpose (no respondent answered it): col", i, "|", full_header(i)[:70], "|", why)
    log("2.0 unused column", "col " + str(i) + " has 0 answers and is not carried into the output: " + why)
covered |= set(UNUSED)
uncovered = [i for i in range(len(RAW)) if i not in covered]
print("source columns used by no output column:", uncovered)
assert not uncovered, "a source column would be dropped silently"

CONFIG = {"drop_rather_not_say": True, "drop_student_unemployed": True, "top_band_midpoint": 150000}
print(CONFIG)

row_log = [("1.6 option-label row moved out of data", len(csv_rows) - 1, len(df))]
def log_rows(step, before, after):
    row_log.append((step, before, after))
    print("[ROWS]", step, ":", before, "->", after, "(removed", before - after, ")")

ok  Country                        single   cols 0          | What Country do you live in?
ok  USState                        single   cols 2          | Which US State or Territory do you live in?
ok  Age                            single   cols 3          | How old are you?
ok  Gender                         single   cols 4          | What is your gender?
ok  YearsCode                      single   cols 5          | How many years of IT/Programming experience do you have?
ok  DevType                        single   cols 6          | Which of the following best describes your occupation?
ok  CompTotal                      salary   cols 7          | Including bonus, what is your annual compensation in USD?
ok  Industry                       single   cols 8          | How would you best describe the industry you currently work 
ok  OrgDevCount                    single   cols 9          | How many developers are employed at your company?
ok  RemoteWork                     single   cols 1

## Gate 3 - Excel date damage (repairs, before any row is dropped)

### Cell 3.1 - Repair damaged band labels and check every column against its expected levels
**Purpose:** Excel turned some band labels into dates or numbers when the file was saved (`2-5` became a date, `11+` became `11`). This cell lists every raw value of each affected column with its repair, applies it, and asserts that **every value left is one of the expected levels**. It runs before any rows are dropped, so the counts below are whole-file counts you can compare with your own checks.
**Decision made:** the repairs are the ones from your notes (dates and bare numbers back to the survey's band labels; `Just me!` -> `1`, `Don't have an account` -> `0`, reputation levels `1000` -> `1000+` and so on). Columns with an empty repair table only get the level check. Numeric Excel serials are decoded below so you can see which date each one is. Every other single-answer column keeps its text exactly as typed.

In [12]:
REP_MAP = {"1000": "1000+", "500": "500+", "1": "1+", "200": "200+", "10000": "10000+", "100": "100+", "5000": "5000+",
           "2000": "2000+", "3000": "3000+", "50": "50+", "Don't have an account": "0"}
REP_ORDER = ["0", "1+", "50+", "100+", "200+", "500+", "1000+", "2000+", "3000+", "5000+", "10000+"]
FIXES = {
    5: ({"6/10/2014": "6-10", "2/5/2014": "2-5", "11": "11+"}, ["<2", "2-5", "6-10", "11+"]),
    9: ({"1/5/2014": "1-5", "6/15/2014": "6-15", "100": "100+"}, ["1-5", "6-15", "16-30", "31-50", "50-100", "100+"]),
    41: ({}, ["Don't know", "<$1,000", "$1,000 - $5,000", "$5,001 - $10,000", "$10,001 - $25,000", "$25,001 - $100,000", ">$100,001"]),
    112: (REP_MAP, REP_ORDER),
}

df_raw = df.copy()   # untouched copy, used by the validation in Gate 8
repaired_cols = []
for idx, (mapping, order) in FIXES.items():
    col = RAW[idx]
    before = df[col].value_counts(dropna=False)
    print("---", idx, "|", full_header(idx)[:90])
    table = pd.DataFrame({"raw_value": ["(blank)" if pd.isna(v) else str(v) for v in before.index], "n_rows": before.values})
    table["repaired_to"] = [mapping.get(v, "(unchanged)") if v != "(blank)" else "(blank - stays blank)" for v in table["raw_value"]]
    print(table.to_string(index=False))
    unused = [k for k in mapping if k not in before.index]
    assert not unused, "repair keys that never occur: " + str(unused)
    for k in mapping:
        if k.isdigit() and int(k) > 30000:
            print("   Excel serial", k, "=", datetime.date(1899, 12, 30) + datetime.timedelta(days=int(k)))
    df[col] = df[col].replace(mapping)
    left = sorted(set(df[col].dropna()) - set(order))
    assert not left, "values outside the expected levels: " + str(left)
    after = df[col].value_counts(dropna=False)
    print("after repair, in the expected order:", {k: int(after.get(k, 0)) for k in order}, "| blank:", int(df[col].isna().sum()))
    if mapping:
        repaired_cols.append(idx)
log("3.1 repairs", "damaged band labels repaired in columns " + str(repaired_cols) + "; all other single-answer columns untouched")

--- 5 | How many years of IT/Programming experience do you have?
raw_value  n_rows           repaired_to
 2/5/2014    2405                   2-5
       11    2036                   11+
6/10/2014    1817                  6-10
       <2    1088           (unchanged)
  (blank)     297 (blank - stays blank)
after repair, in the expected order: {'<2': 1088, '2-5': 2405, '6-10': 1817, '11+': 2036} | blank: 297
--- 9 | How many developers are employed at your company?
raw_value  n_rows           repaired_to
 1/5/2014    2131                   1-5
      100    1647                  100+
  (blank)    1246 (blank - stays blank)
6/15/2014    1104                  6-15
    16-30     619           (unchanged)
   50-100     453           (unchanged)
    31-50     443           (unchanged)
after repair, in the expected order: {'1-5': 2131, '6-15': 1104, '16-30': 619, '31-50': 443, '50-100': 453, '100+': 1647} | blank: 1246
--- 41 | What is your budget for outside expenditures (hardware, software, con

## Gate 4 - Rows: junk, salary

### Cell 4.1 - Junk rows: propose
**Purpose:** respondents who answered nothing except Country/State (US State is routing, not an answer). **Decision made:** none yet.

In [13]:
SALARY_IDX = [spec for kind, name, status, spec, expect, note in ITEMS if kind == "salary"][0]
SAL = RAW[SALARY_IDX]
other_cols = [c for c in df.columns if c not in (RAW[0], RAW[1])]
answered_other = df[other_cols].notna().sum(axis=1)
is_junk = answered_other == 0
print("rows the rule would remove:", int(is_junk.sum()), "| of those with a salary band:", int((is_junk & df[SAL].notna()).sum()))

rows the rule would remove: 249 | of those with a salary band: 0


### Cell 4.2 - Junk rows: apply
**Purpose / decision:** drop them (none has a salary). Counts logged.

In [14]:
rows_before = len(df)
df = df.loc[~is_junk].copy()
log_rows("4.2 junk rows (only Country/State answered)", rows_before, len(df))
log("4.2 junk rows", str(rows_before - len(df)) + " rows removed")

[ROWS] 4.2 junk rows (only Country/State answered) : 7643 -> 7394 (removed 249 )
[LOG] 4.2 junk rows - 249 rows removed


### Cell 4.3 - Salary band table
**Purpose:** the answer is a text band, not a number. This table fixes the order, the bounds and the midpoint used to make an annual-USD number, and lists the two non-amount answers.
**Decision made:** `CompTotal` keeps the original band text; `ConvertedCompYearly` is the band midpoint in annual USD (the question says "including bonus"). The open-ended `>$140,000` band takes `CONFIG["top_band_midpoint"]` (same value as 2011, so the years stay comparable). Pay before 2019 is base salary only in the shared schema notes, so the band-midpoint construction is a 2011-2015 convention.

In [15]:
STUDENT_LABEL = "Student / Unemployed"
RNS_LABEL = "Rather not say"
BANDS = [
    ("<$20,000", 0, 20000, 10000), ("$20,000 - $40,000", 20000, 40000, 30000), ("$40,000 - $60,000", 40000, 60000, 50000),
    ("$60,000 - $80,000", 60000, 80000, 70000), ("$80,000 - $100,000", 80000, 100000, 90000),
    ("$100,000 - $120,000", 100000, 120000, 110000), ("$120,000 - $140,000", 120000, 140000, 130000),
    (">$140,000", 140000, None, CONFIG["top_band_midpoint"]),
]
observed = set(df[SAL].dropna().unique())
known = set(b[0] for b in BANDS) | {STUDENT_LABEL, RNS_LABEL}
assert observed <= known, sorted(observed - known)
counts = df[SAL].value_counts()
print(pd.DataFrame([[b[0], int(counts.get(b[0], 0)), b[1], b[2], b[3]] for b in BANDS] +
                   [[STUDENT_LABEL, int(counts.get(STUDENT_LABEL, 0)), None, None, None], [RNS_LABEL, int(counts.get(RNS_LABEL, 0)), None, None, None]],
                   columns=["answer", "n_rows", "lower", "upper", "midpoint_used"]).to_string(index=False))
print("missing salary answer:", int(df[SAL].isna().sum()))

              answer  n_rows    lower    upper  midpoint_used
            <$20,000     970      0.0  20000.0        10000.0
   $20,000 - $40,000     746  20000.0  40000.0        30000.0
   $40,000 - $60,000     794  40000.0  60000.0        50000.0
   $60,000 - $80,000     697  60000.0  80000.0        70000.0
  $80,000 - $100,000     587  80000.0 100000.0        90000.0
 $100,000 - $120,000     386 100000.0 120000.0       110000.0
 $120,000 - $140,000     199 120000.0 140000.0       130000.0
           >$140,000     244 140000.0      NaN       150000.0
Student / Unemployed    1147      NaN      NaN            NaN
      Rather not say    1130      NaN      NaN            NaN
missing salary answer: 494


### Cell 4.4 - Drop rows with no usable salary
**Purpose / decision:** drop blank salary, then (per `CONFIG`) `Rather not say` and `Student / Unemployed`: none of the three is an amount. Counts logged after each step.

In [16]:
rows_before = len(df)
df = df.loc[df[SAL].notna()].copy()
log_rows("4.4 salary answer blank", rows_before, len(df))
if CONFIG["drop_rather_not_say"]:
    rows_before = len(df)
    df = df.loc[df[SAL] != RNS_LABEL].copy()
    log_rows("4.4 'Rather not say'", rows_before, len(df))
if CONFIG["drop_student_unemployed"]:
    rows_before = len(df)
    df = df.loc[df[SAL] != STUDENT_LABEL].copy()
    log_rows("4.4 Student / Unemployed", rows_before, len(df))
assert set(df[SAL].unique()) <= set(b[0] for b in BANDS)
log("4.4 salary rows", "rows without a salary amount removed; " + str(len(df)) + " remain")

[ROWS] 4.4 salary answer blank : 7394 -> 6900 (removed 494 )
[ROWS] 4.4 'Rather not say' : 6900 -> 5770 (removed 1130 )
[ROWS] 4.4 Student / Unemployed : 5770 -> 4623 (removed 1147 )
[LOG] 4.4 salary rows - rows without a salary amount removed; 4623 remain


## Gate 5 - Multi-answer questions stored as lists

### Cell 5.1 - Helper and all list blocks
**Purpose:** these questions have one column per option. 2016+ files store the same kind of question as one `;`-separated text cell, so every year is written that way and pooled multi-hot encoding is later one function for every year.
**Decisions made:** option labels come from row 0; `Other (please specify)` becomes `Other` (its typed text is kept in its own free-text column, not lost); every option cell must equal its label (asserted); labels are asserted to contain no `;`; a respondent who ticked nothing gets an empty cell (NaN), not an empty string. Options are listed in questionnaire order. Language blocks are handled in Gate 6.

In [17]:
def block_to_list(frame, a, b):
    cols = RAW[a:b]
    labels = [label(i) for i in range(a, b)]
    is_other = [str(l).lower().startswith("other (please specify)") for l in labels]
    nice = ["Other" if o else l for l, o in zip(labels, is_other)]
    assert all(";" not in l for l in nice), "an option label contains ';'"
    for col, l, o in zip(cols, labels, is_other):
        if not o:
            assert (frame[col].dropna() == l).all(), "cell text differs from the option label in " + col
    def join_row(values):
        picked = [n for n, v in zip(nice, values) if pd.notna(v)]
        return ";".join(picked) if picked else np.nan
    series = frame[cols].apply(lambda r: join_row(r.values), axis=1)
    return series, nice

LISTS = {}
for kind, name, status, spec, expect, note in ITEMS:
    if kind != "list":
        continue
    series, nice = block_to_list(df, spec[0], spec[1])
    LISTS[name] = series
    per_row = df[RAW[spec[0]:spec[1]]].notna().sum(axis=1)
    print("---", name, "| cols", spec[0], "to", spec[1] - 1, "| blank:", int(series.isna().sum()))
    print("   options and rows that ticked each:", {n: int(df[c].notna().sum()) for n, c in zip(nice, RAW[spec[0]:spec[1]])})
    print("   rows by number of options ticked:", per_row.value_counts().sort_index().to_dict())
    print("   example:", series.dropna().iloc[0])
log("5.1 list blocks", str(len(LISTS)) + " multi-answer blocks stored as ';' lists: " + ", ".join(LISTS))

--- MobilePlatformsSupported | cols 13 to 19 | blank: 514
   options and rows that ticked each: {'iPhone': 1621, 'iPad': 1351, 'Android Phone': 1616, 'Android Tablet': 1222, 'Windows Phone': 559, 'Other': 247, 'No mobile app': 1994}
   rows by number of options ticked: {0: 514, 1: 2332, 2: 568, 3: 230, 4: 563, 5: 299, 6: 114, 7: 3}
   example: iPhone;Android Phone
--- PurchaseInvolvement | cols 29 to 33 | blank: 239
   options and rows that ticked each: {'I can recommend or request products': 2301, "I influence purchasing decisions, but don't have final approval": 1476, 'I have a discretionary budget at my disposal': 267, 'I can buy anything I want without asking anyone': 398, 'No Involvement': 1231}
   rows by number of options ticked: {0: 239, 1: 3244, 2: 1017, 3: 100, 4: 20, 5: 3}
   example: I can recommend or request products
--- PurchaseTypes | cols 34 to 40 | blank: 1416
   options and rows that ticked each: {'Hardware': 1428, 'Servers': 888, 'Software': 2811, 'User Equipment: M

## Gate 6 - Language proficiency

### Cell 6.1 - Look at the block
**Purpose:** print the tick-box labels and the write-in volume before deciding anything.
**Decision made:** none.

In [18]:
LANG_SPEC = [spec for kind, name, status, spec, expect, note in ITEMS if kind == "lang"][0]
a, b = LANG_SPEC
lang_cols = RAW[a:b]
lang_labels = [label(i) for i in range(a, b)]
for i, lab in zip(range(a, b), lang_labels):
    print("col", i, "->", lab, "| ticked:", int(df[RAW[i]].notna().sum()))
WRITEIN_COL = lang_cols[-1]
assert lang_labels[-1].lower().startswith("other (please specify)")
writein = df[WRITEIN_COL].dropna()
print()
print("write-in answers:", len(writein), "| distinct as typed:", writein.nunique(), "| distinct lowercased:", writein.str.lower().str.strip().nunique())
print("most common (lowercased):")
print(writein.str.lower().str.strip().value_counts().head(15).to_string())

col 42 -> C | ticked: 656
col 43 -> C++ | ticked: 808
col 44 -> C# | ticked: 1717
col 45 -> Java | ticked: 1515
col 46 -> JavaScript | ticked: 2603
col 47 -> Node.js | ticked: 456
col 48 -> Objective-C | ticked: 462
col 49 -> PHP | ticked: 1036
col 50 -> Python | ticked: 899
col 51 -> Ruby | ticked: 410
col 52 -> SQL | ticked: 2444
col 53 -> Other (please specify) | ticked: 944

write-in answers: 944 | distinct as typed: 447 | distinct lowercased: 372
most common (lowercased):
Unnamed: 53
vb.net        66
scala         57
perl          54
r             33
delphi        32
groovy        25
matlab        23
vba           23
bash          20
jquery        16
coldfusion    16
powershell    15
go            15
f#            14
vb            13


### Cell 6.2 - What the tick-boxes become (canonical names)
**Purpose:** a tick-box and a write-in must end up with the same name. This table maps each tick-box to a language, a tool, or "none".
**Decisions made (review me):**
- `CSS` and `HTML5` both become **`HTML/CSS`** (a respondent who ticked both gets it once).
- `None` means no languages: it adds nothing to the list and the count is 0 (not missing).
- **Tools are not languages.** If a tick-box is a tool (`jQuery`, `Node.js` in 2013) it goes to the separate column `ToolsHaveWorkedWith`, not into the language list or count. Both `jQuery` and `JQuery` boxes exist in 2013 and are the same option.
- Canonical names follow the 2021+ style (`Bash/Shell`, `HTML/CSS`, `Objective-C`).

In [19]:
TICK_MAP = {"C": ("lang", "C"), "C++": ("lang", "C++"), "C#": ("lang", "C#"), "Java": ("lang", "Java"), "JavaScript": ("lang", "JavaScript"),
            "Node.js": ("tool", "Node.js"), "Objective-C": ("lang", "Objective-C"), "PHP": ("lang", "PHP"), "Python": ("lang", "Python"),
            "Ruby": ("lang", "Ruby"), "SQL": ("lang", "SQL")}
print(pd.DataFrame([[k, v[0], v[1]] for k, v in TICK_MAP.items()], columns=["tick-box label (row 0)", "goes to", "canonical name"]).to_string(index=False))
assert set(lang_labels[:-1]) == set(TICK_MAP), sorted(set(lang_labels[:-1]) ^ set(TICK_MAP))

tick-box label (row 0) goes to canonical name
                     C    lang              C
                   C++    lang            C++
                    C#    lang             C#
                  Java    lang           Java
            JavaScript    lang     JavaScript
               Node.js    tool        Node.js
           Objective-C    lang    Objective-C
                   PHP    lang            PHP
                Python    lang         Python
                  Ruby    lang           Ruby
                   SQL    lang            SQL


### Cell 6.3 - Alias dictionary for the write-ins
**Purpose:** turn free text into canonical names. People typed several languages per box, in mixed case, with typos, abbreviations and commentary.
**Method:** each rule is a pattern searched anywhere in the lowercased text, and a match is *removed* before the next rule runs, so `vb.net` is not also counted as `vb`. More specific rules come first.
**This is the 2011 dictionary plus additions** needed by the 2012/2013 write-ins (CoffeeScript, TypeScript, Dart, Go, Racket, Octave, XQuery, XSLT, Apex, PL/pgSQL, CUDA, ...). A rule that is shared by all years keeps the same canonical name, so the years pool cleanly.
**Decisions made (review me):**
- **Languages are counted; tools are not.** Frameworks, libraries, databases and platforms (jQuery, Django, Rails, MongoDB, MySQL, ASP.NET, Node.js...) are recognised and stored in `ToolsHaveWorkedWith`, but kept out of the language list and count. Databases are tools: a respondent who typed `MySQL` is not counted as knowing `SQL`.
- `Ruby on Rails` counts `Ruby` (Rails implies Ruby) and records `Rails` as a tool.
- **Visual Basic stays split three ways:** `VB.NET`, `Visual Basic 6` and plain `Visual Basic`.
- One-letter names (`R`, `C`, `D`, `J`, `Go`) only count when that is the whole phrase.
- Anything left over is kept as `unrecognized` for audit and never counted. Real languages typed that this dictionary does not know are therefore **not** counted, so counts are a slight undercount for rare languages.

In [20]:
def _name_regex(name):
    return r"\b" + re.escape(name.lower()).replace(r"\ ", r"\s+") + r"\b"

LANG_RULES = [
    ("VB.NET", r"visual\s*basic\s*\.?\s*net|\bvb\s*\.?\s*net"),
    ("VBScript", r"\bvb\s*script|\bvbs\b"),
    ("VBA", r"\bvba\b|visual\s*basic\s+for\s+applications"),
    ("Visual Basic 6", r"\bvb\s*6\b|visual\s*basic\s*6|classic\s*vb"),
    ("Visual Basic", r"visual\s*basic|\bvb\b"),
    ("PL/SQL", r"\bpl\s*/\s*sql"),
    ("SQL", r"\bsql\b"),
    ("Objective-C", r"\bobj(?:ective|ectve)?[\s\-]*c\b"),
    ("C#", r"\bc\s?#"),
    ("C++", r"\bc\s?\+\+|\bcpp\b"),
    ("F#", r"\bf\s?#"),
    ("Java", r"\bjava\b"),
    ("JavaScript", r"\bjavascript\b|\becmascript\b"),
    ("PHP", r"\bphp\b"),
    ("Python", r"\bpython\b"),
    ("Ruby", r"\bruby\b"),
    ("Perl", r"\bperl\b"),
    ("HTML/CSS", r"\bx?html5?\b|\bcss3?\b"),
    ("PowerShell", r"powershell"),
    ("Bash/Shell", r"\bbash\b|\bshell?(?:\s+script\w*)?\b|\bksh\b|\bzsh\b|\bcsh\b|\bsh\b"),
    ("Assembly", r"(?:x86\s+|arm\s+|z/)?assembl(?:y|er)\b(?:\s+language)?"),
    ("Delphi", r"delphi|object\s*pascal|\boo-pascal\b"),
    ("Pascal", r"\bpascal\b"),
    ("ActionScript", r"actionscrip\w*(?:\s*3(?:\.0)?)?|\bas[123](?:-[123])?\b"),
    ("ColdFusion", r"coldfusion|\bcfml\b"),
    ("Lisp", r"\blisp\b"),
    ("MEL", r"\bmel\b"),
    # ---- additions for 2012/2013 ----
    ("PL/pgSQL", r"\bpl\s*/?\s*pgsql\b"),
    ("PL/SQL", r"\bplsql\b"),
    ("CoffeeScript", r"\bcoff?e+s+cript\b|\bcoffee\s+script\b"),
    ("TypeScript", r"\btypescript\b"),
    ("Dart", r"\bdart\b"),
    ("Go", r"\bgolang\b|\bgoogle\s+go\b"),
    ("Racket", r"\bracket\b"),
    ("Octave", r"\boctave\b"),
    ("XQuery", r"\bxquery\b"),
    ("XSLT", r"\bxslt?\b"),
    ("Apex", r"\bapex\b"),
    ("Emacs Lisp", r"\belisp\b"),
    ("PostScript", r"\bpostscript\b"),
    ("Haxe", r"\bhaxe\b"),
    ("QML", r"\bqml\b"),
    ("HLSL", r"\bhlsl\b"),
    ("CUDA", r"\bcuda\b"),
    ("SML", r"\bsml\b"),
    ("APL", r"\bapl\b"),
    ("Pawn", r"\bpawn\b"),
    ("BrightScript", r"\bbrightscript\b"),
    ("RPG", r"\brpgle\b"),
    ("FoxPro", r"visual\s*foxpro|\bvfp\b"),
    ("Progress ABL", r"progress\s+(?:openedge|4gl|abl)|\bopenedge\b"),
    ("Assembly", r"\basm\b|\bnasm\b"),
    ("ActionScript", r"\baction\s+script(?:\s*3(?:\.0)?)?\b"),
    ("ColdFusion", r"\bcold\s+fusion\b"),
    ("HTML/CSS", r"\bhtml4\b"),
    ("Pascal", r"free\s*pascal"),
    ("Julia", r"\bjulia\b"),
    ("Coq", r"\bcoq\b"),
    ("Stata", r"\bstata\b"),
    ("SystemVerilog", r"\bsystemverilog\b"),
    ("GLSL", r"\bglsl\b"),
]
RARE_LANGS = ["Scheme", "Clojure", "Haskell", "Scala", "Erlang", "Lua", "Groovy", "OCaml", "Prolog", "Fortran", "COBOL", "Ada",
              "Tcl", "AWK", "Smalltalk", "Eiffel", "Forth", "Rexx", "MATLAB", "Mathematica", "Maple", "IDL", "SAS", "RPG", "ABAP",
              "PowerBuilder", "FoxPro", "LabVIEW", "AppleScript", "VHDL", "Verilog", "AutoHotkey", "AutoIt", "LSL", "Avisynth",
              "KRL", "Wasabi", "GeneXus", "LotusScript", "UniBasic", "UniQuery", "Nemerle", "Gosu", "Vala", "Brainfuck", "Befunge",
              "LOLCODE", "RealBasic", "CCScript", "DataFlex", "WebFocus", "Linc 4GL", "Forte 4GL", "Progress ABL",
              "JMP Scripting Language", "XSLT", "Lansa", "BASIC"]
LANG_RULES += [(n, _name_regex(n)) for n in RARE_LANGS]

TOOL_RULES = [
    ("ASP.NET", r"\basp\.net(?:-mvc)?"), ("ASP", r"(?:classic\s+)?\basp\b(?:\s*(?:classic|3\.0))?"), (".NET", r"(?<!\w)\.net\b"),
    ("jQuery", r"\bjquery\b"), ("JSP", r"\bjsp\b"), ("Symfony", r"\bsymfony\b"), ("Silverlight", r"\bsilverlight\b"),
    ("WPF", r"\bwpf\b"), ("WCF", r"\bwcf\b"), ("Entity Framework", r"entity-framework"), ("Grails", r"\bgrails\b"),
    ("Cocoa", r"\bcocoa\b"), ("Android", r"\bandroid\b"), ("Flex", r"(?:adobe\s+)?\bflex\b"), ("Flash", r"\bflash\b"),
    ("Excel", r"\bexcel\b"), ("NumPy", r"\bnumpy\b"), ("LaTeX", r"\blatex\b"), ("XML", r"\bxml\b"), ("XAML", r"\bxaml\b"),
    ("X3D", r"\bx3d\b"), ("XPath", r"\bxpath\b"), ("sed", r"\bsed\b"),
    # ---- additions for 2012/2013 ----
    (".NET", r"\bdot\s*net\b"), ("Node.js", r"\bnode\.?js\b|\bnode\b"), ("Django", r"\bdjango\b"), ("Rails", r"\b(?:on\s+)?rails\b"),
    ("Spring", r"\bspring\b"), ("Hibernate", r"\bhibernate\b"), ("Struts", r"\bstruts2?\b"), ("MVC", r"\bmvc\b"),
    ("Dojo", r"\bdojo\b"), ("Ext JS", r"\bext\s?js\b"), ("AngularJS", r"\bangular(?:\s?js)?\b"), ("Backbone.js", r"\bbackbone(?:\.js)?\b"),
    ("Knockout.js", r"\bknockout(?:\.?js)?\b"), ("Underscore.js", r"\bunderscore(?:\.js)?\b"), ("D3.js", r"\bd3(?:\.js)?\b"),
    ("MooTools", r"\bmootools\b"), ("GWT", r"\bgwt\b"), ("JSF", r"\bjsf\b"), ("YUI", r"\byui\b"), ("Zend Framework", r"\bzend(?:\s+framework)?\b"),
    ("Play Framework", r"\bplay\s+framework(?:\s*\d)?\b"), ("Drupal", r"\bdrupal\b"), ("Magento", r"\bmagento\b"), ("Razor", r"\brazor\b"),
    ("LINQ", r"\blinq\b"), ("Qt", r"\bqt\b"), ("AJAX", r"\bajax\b"), ("JSON", r"\bjson\b"), ("SSIS/SSRS", r"\bss[ir]s\b"),
    ("Puppet", r"\bpuppet\b"), ("Git", r"\bgit\b"), ("nginx", r"\bnginx\b"), ("MongoDB", r"\bmongo(?:db)?\b"), ("MySQL", r"\bmysql\b"),
    ("PostgreSQL", r"\bpostgres(?:ql)?\b"), ("Redis", r"\bredis\b"), ("CouchDB", r"\bcouchdb\b"), ("Neo4j", r"\bneo4j\b"),
    ("Oracle", r"\boracle\b"), ("Solr", r"\bsolr\b"), ("Elasticsearch", r"\belasticsearch\b"), ("Sass/Less", r"\bsass\b|\bscss\b"),
    ("PhoneGap", r"\bphonegap\b"), ("SharePoint", r"\bsharepoint\b"), ("jQuery", r"\bjquery\b"),
    ("Hadoop", r"\bhadoop\b"), ("Spark", r"\bspark\b"), ("Cassandra", r"\bcassandra\b"), ("Gradle", r"\bgradle\b"),
]
EXACT_LANG = {"r": "R", "d": "D", "j": "J", "go": "Go", "c": "C"}
EXACT_TOOL = {"make": "Make", "less": "Sass/Less"}
# two answers where commentary contains language-like words; decided by hand and shown in Cell 5.5
MANUAL = {
    "c/al (microsoft dynamics nav)": {"C/AL"},
    "guys you forget scala in exciting techno. shame on you": set(),
"css is not a programming language.": set(),
          "vb (why the fuck was this not a choice? c# snobs?)": {"Visual Basic"}}

COMPILED_LANG = [(n, re.compile(p)) for n, p in LANG_RULES]
COMPILED_TOOL = [(n, re.compile(p)) for n, p in TOOL_RULES]

def parse_writein(text):
    s = str(text).lower().strip().replace("’", "'")
    if s in MANUAL:
        return set(MANUAL[s]), set(), ["(manual override)"]
    langs, tools = set(), set()
    for name, rx in COMPILED_LANG:
        if rx.search(s):
            langs.add(name)
            s = rx.sub(" | ", s)
    for name, rx in COMPILED_TOOL:
        if rx.search(s):
            tools.add(name)
            s = rx.sub(" | ", s)
    leftovers = []
    for phrase in re.split(r"[|,;/&+()\[\]:!?]+", s):
        p = re.sub(r"[^a-z0-9#+.\- ]", "", phrase).strip(" .-")
        if not p:
            continue
        if p in EXACT_LANG:
            langs.add(EXACT_LANG[p])
        elif p in EXACT_TOOL:
            tools.add(EXACT_TOOL[p])
        else:
            leftovers.append(p)
    return langs, tools, leftovers

alias_table = pd.DataFrame([[n, "language", p] for n, p in LANG_RULES] + [[n, "tool/markup (not counted)", p] for n, p in TOOL_RULES] +
                           [[v, "language (exact phrase only)", "^" + k + "$"] for k, v in EXACT_LANG.items()],
                           columns=["canonical_name", "kind", "pattern"])
print("alias rules:", len(alias_table), "| language names:", int((alias_table.kind == "language").sum()), "| tool names:", int(alias_table.kind.str.startswith("tool").sum()))
for test in ["vb.net, vb6", "coffeescript, golang", "ruby on rails, mysql", "node.js and c", "x86 asm", "plsql"]:
    print(repr(test), "->", parse_writein(test))

alias rules: 196 | language names: 119 | tool names: 72
'vb.net, vb6' -> ({'Visual Basic 6', 'VB.NET'}, set(), [])
'coffeescript, golang' -> ({'Go', 'CoffeeScript'}, set(), [])
'ruby on rails, mysql' -> ({'Ruby'}, {'MySQL', 'Rails'}, [])
'node.js and c' -> (set(), {'Node.js'}, ['and c'])
'x86 asm' -> ({'Assembly'}, set(), ['x86'])
'plsql' -> ({'PL/SQL'}, set(), [])


### Cell 6.4 - Parse every write-in and show the result
**Purpose:** apply the dictionary to all write-ins in the retained rows. The table shows, for the canonical names found, how many respondents wrote each one, and the tools recognised.
**Decision made:** none; this is the audit step.

In [21]:
parsed = {}
for idx, text in writein.items():
    parsed[idx] = parse_writein(text)

lang_counts, tool_counts = {}, {}
for idx, (langs, tools, left) in parsed.items():
    for name in langs:
        lang_counts[name] = lang_counts.get(name, 0) + 1
    for name in tools:
        tool_counts[name] = tool_counts.get(name, 0) + 1
print("languages recognised in write-ins (respondents):")
print(pd.Series(lang_counts).sort_values(ascending=False).to_frame("respondents").T.to_string())
print()
print("tools recognised in write-ins (not counted as languages):")
print(pd.Series(tool_counts).sort_values(ascending=False).to_frame("respondents").T.to_string())
print()
n_none = sum(1 for v in parsed.values() if not v[0])
print("write-ins with no language recognised:", n_none, "of", len(parsed), "| of those with at least one tool:", sum(1 for v in parsed.values() if not v[0] and v[1]))

languages recognised in write-ins (respondents):
             VB.NET  Perl  Scala  Bash/Shell   R  VBA  Delphi  Groovy  MATLAB  HTML/CSS  Go  Visual Basic  PowerShell  ColdFusion  Clojure  Haskell  ActionScript  Lua  F#  Assembly  VBScript  CoffeeScript  Tcl  Visual Basic 6  PL/SQL  XSLT  Fortran  Lisp  Erlang  COBOL  SAS  TypeScript  SQL  ABAP  OCaml  CUDA  D  Verilog  LabVIEW  VHDL  PHP  XQuery  Prolog  BASIC  AutoIt  Haxe  LotusScript  Rexx  Stata  Scheme  IDL  SystemVerilog  Progress ABL  GLSL  AppleScript  AWK  Python  RPG  Julia  C/AL  Smalltalk  PostScript  Ada  Dart  C  Coq  Pascal  DataFlex  Eiffel  QML  Octave  Mathematica  JavaScript  Forth
respondents      86    75     72          50  43   38      37      35      33        26  23            20          20          19       17       16            16   15  15        14        12            12   12              11      11     9        8     8       8      7    7           7    5     4      4     4  4        4        4     4   

### Cell 6.5 - Review the odd ones
**Purpose:** two checks that the parser did not invent languages. (1) every write-in longer than 25 characters or with punctuation that signals commentary, with its parse (first 60 shown); (2) the most common leftover phrases, in case a real language was missed.
**Decision made:** none. Any miss is fixed by adding a rule in Cell 6.3, not by editing data.

In [22]:
long_ones = [(idx, t) for idx, t in writein.items() if len(t) > 25 or "?" in t or "!" in t]
print("commentary-style write-ins:", len(long_ones), "(first 60 shown)")
for idx, t in long_ones[:60]:
    langs, tools, left = parsed[idx]
    print("row", idx, "|", repr(t[:100]))
    print("      languages:", sorted(langs), "| tools:", sorted(tools), "| leftover:", left[:4])
print()
leftover_counts = {}
for langs, tools, left in parsed.values():
    for p in left:
        leftover_counts[p] = leftover_counts.get(p, 0) + 1
print("most common leftover phrases:")
print(pd.Series(leftover_counts).sort_values(ascending=False).head(40).to_string())

commentary-style write-ins: 33 (first 60 shown)
row 400 | 'VB.Net (nice snark on your part)'
      languages: ['VB.NET'] | tools: [] | leftover: ['nice snark on your part']
row 445 | 'VB.Net, ActionScript 2.0, ActionScript 3.0'
      languages: ['ActionScript', 'VB.NET'] | tools: [] | leftover: ['2.0']
row 561 | 'Bash script and Batch script'
      languages: ['Bash/Shell'] | tools: [] | leftover: ['script and batch script']
row 1074 | 'Scala, but still in learning phase'
      languages: ['Scala'] | tools: [] | leftover: ['but still in learning phase']
row 1098 | 'RyuQ, Erlang, Guile/Scheme, Haskell'
      languages: ['Erlang', 'Haskell', 'Scheme'] | tools: [] | leftover: ['ryuq', 'guile']
row 1352 | 'CSS, HTML, ActionScript, Flash'
      languages: ['ActionScript', 'HTML/CSS'] | tools: ['Flash'] | leftover: []
row 1829 | 'VBA (seriously!)'
      languages: ['VBA'] | tools: [] | leftover: ['seriously']
row 1939 | "Optics Raytracing with Breault's ASAP"
      languages: [] | tools: [] 

### Cell 6.6 - Build `LanguageHaveWorkedWith`, the count, and `ToolsHaveWorkedWith`
**Purpose:** combine the tick-boxes and the parsed write-ins into one canonical list per respondent, plus a count.
**Decisions made:**
- The language list is the **distinct** canonical names, sorted alphabetically, joined with `;`.
- `LanguageHaveWorkedWith_count` is the length of that list. It is `0` only for people who ticked `None` with no language. It is empty (NaN, meaning unknown) for people who left the block blank **and** for people whose only answer was a write-in with no recognisable language (for example "various"). Someone who ticked `None` and also named a language keeps the language.
- A name in both the tick-boxes and the write-in is counted once.
- `ToolsHaveWorkedWith` is the same kind of `;` list for tools. An empty cell there only means no tool was recognised; it does not mean unknown. In 2012 there are no tool tick-boxes, so the column comes from write-ins only.
- Caution for the pooled model: the number of tick-boxes differs every year (2011: 12, 2012: 14, 2013: 13, 2021+: 40+), so a count of 4 does not mean the same in every year. Compare counts within a year, or let the model see `survey_year`.

In [23]:
tick_cols = lang_cols[:-1]
tick_labels = lang_labels[:-1]
final_lists, final_counts, final_tools, none_flags = [], [], [], []
for idx in df.index:
    names, tools = set(), set()
    none_ticked = False
    for col, lab in zip(tick_cols, tick_labels):
        if pd.notna(df.at[idx, col]):
            kind, canon = TICK_MAP[lab]
            if kind == "none":
                none_ticked = True
            elif kind == "lang":
                names.add(canon)
            else:
                tools.add(canon)
    if idx in parsed:
        names |= parsed[idx][0]
        tools |= parsed[idx][1]
    final_lists.append(";".join(sorted(names, key=str.lower)) if names else np.nan)
    final_tools.append(";".join(sorted(tools, key=str.lower)) if tools else np.nan)
    if names:
        final_counts.append(len(names))
    elif none_ticked:
        final_counts.append(0)
    else:
        final_counts.append(np.nan)
    none_flags.append(none_ticked)

language_list = pd.Series(final_lists, index=df.index, dtype=object)
language_count = pd.Series(final_counts, index=df.index, dtype=float)
tools_list = pd.Series(final_tools, index=df.index, dtype=object)
none_ticked = pd.Series(none_flags, index=df.index)
block_answered = df[lang_cols].notna().any(axis=1)
print("rows:", len(df), "| count blank:", int(language_count.isna().sum()),
      "(block unanswered:", int((~block_answered).sum()), "| answered but only an unrecognisable write-in:", int((block_answered & language_list.isna() & ~none_ticked).sum()), ")")
print("'None' ticked:", int(none_ticked.sum()), "| of those, rows that also have a language:", int((none_ticked & language_list.notna()).sum()), "| count = 0 rows:", int((language_count == 0).sum()))
print("count distribution:")
print(language_count.value_counts().sort_index().to_string())
print("mean count: %.2f" % language_count.mean())
print("rows with at least one tool:", int(tools_list.notna().sum()))
log("6.6 languages", "tick-boxes + parsed write-ins -> LanguageHaveWorkedWith list, count and ToolsHaveWorkedWith")

rows: 4623 | count blank: 313 (block unanswered: 303 | answered but only an unrecognisable write-in: 10 )
'None' ticked: 0 | of those, rows that also have a language: 0 | count = 0 rows: 0
count distribution:
1.0      469
2.0      978
3.0     1474
4.0      781
5.0      358
6.0      160
7.0       51
8.0       24
9.0        6
10.0       8
11.0       1
mean count: 3.11
rows with at least one tool: 590
[LOG] 6.6 languages - tick-boxes + parsed write-ins -> LanguageHaveWorkedWith list, count and ToolsHaveWorkedWith


### Cell 6.7 - Sanity checks and audit file
**Purpose:** confirm no name is spelled two ways, the list round-trips to the count, and write out an audit file so every parsed write-in can be checked by eye.
**Decision made:** none.

In [24]:
all_names = sorted({n for s in language_list.dropna() for n in s.split(";")}, key=str.lower)
print("distinct canonical language names in the column:", len(all_names))
print(all_names)
lowered = {}
for n in all_names:
    lowered.setdefault(n.lower(), []).append(n)
assert all(len(v) == 1 for v in lowered.values()), "two spellings of the same name"
tool_names = sorted({n for s in tools_list.dropna() for n in s.split(";")}, key=str.lower)
print("distinct tool names:", len(tool_names))
print(tool_names)
overlap = set(all_names) & set(tool_names)
assert not overlap, "a name is both a language and a tool: " + str(overlap)
for idx in df.index:
    s = language_list[idx]
    n = 0 if pd.isna(s) else len(s.split(";"))
    c = language_count[idx]
    assert (pd.isna(c) and pd.isna(s)) or c == n, "count does not match list at row " + str(idx)
print("count matches list length for every row")

audit = pd.DataFrame({"source_row": list(parsed.keys()), "as_typed": [writein[i] for i in parsed],
                      "languages_found": [";".join(sorted(parsed[i][0])) for i in parsed],
                      "tools_not_counted": [";".join(sorted(parsed[i][1])) for i in parsed],
                      "unrecognized_text": [" | ".join(parsed[i][2]) for i in parsed]})
audit.to_csv("lang_writein_audit_2014.csv", index=False)
alias_table.to_csv("language_alias_table_v2.csv", index=False)
print("saved lang_writein_audit_2014.csv (%d rows) and language_alias_table_v2.csv (%d rules)" % (len(audit), len(alias_table)))

distinct canonical language names in the column: 79
['ABAP', 'ActionScript', 'Ada', 'AppleScript', 'Assembly', 'AutoIt', 'AWK', 'Bash/Shell', 'BASIC', 'C', 'C#', 'C++', 'C/AL', 'Clojure', 'COBOL', 'CoffeeScript', 'ColdFusion', 'Coq', 'CUDA', 'D', 'Dart', 'DataFlex', 'Delphi', 'Eiffel', 'Erlang', 'F#', 'Forth', 'Fortran', 'GLSL', 'Go', 'Groovy', 'Haskell', 'Haxe', 'HTML/CSS', 'IDL', 'Java', 'JavaScript', 'Julia', 'LabVIEW', 'Lisp', 'LotusScript', 'Lua', 'Mathematica', 'MATLAB', 'Objective-C', 'OCaml', 'Octave', 'Pascal', 'Perl', 'PHP', 'PL/SQL', 'PostScript', 'PowerShell', 'Progress ABL', 'Prolog', 'Python', 'QML', 'R', 'Rexx', 'RPG', 'Ruby', 'SAS', 'Scala', 'Scheme', 'Smalltalk', 'SQL', 'Stata', 'SystemVerilog', 'Tcl', 'TypeScript', 'VB.NET', 'VBA', 'VBScript', 'Verilog', 'VHDL', 'Visual Basic', 'Visual Basic 6', 'XQuery', 'XSLT']
distinct tool names: 44
['.NET', 'Android', 'AngularJS', 'ASP', 'ASP.NET', 'Backbone.js', 'Cassandra', 'D3.js', 'Django', 'Dojo', 'Elasticsearch', 'Excel', '

## Gate 7 - Headers and assembly

### Cell 7.1 - Header map
**Purpose:** rename every column to the shared name, and record the original header, kind, status and a note.
**Decisions made (review me):** the **confirmed** names are the ones in your shared list (`Country`, `Age`, `YearsCode`, `OrgSize`, `DevType`, `LanguageHaveWorkedWith`, `OpSys`, `JobSat`, `CompTotal`, `ConvertedCompYearly`). I could not see your Canonical Mapping spreadsheet, so every other name is **proposed**: check it against the sheet and change it in `ITEMS` (Cell 2.0). `ResponseId` is the row number in the 2014 file, since the file has no respondent id. The `kind` column says how each column is stored, so you can decide what to drop before modelling (`free_text` columns are raw typed text).

In [25]:
FIXED_IDX = set(FIXES) | set()
rows_hm = []
def span_text(spec):
    first = spec[0] if isinstance(spec, tuple) else spec
    s = full_header(first)
    if isinstance(spec, tuple):
        s = RAW[PARENT[first]] + " (cols " + str(spec[0]) + "-" + str(spec[1] - 1) + ")"
    return s
for kind, name, status, spec, expect, note in ITEMS:
    if kind == "id":
        rows_hm.append((name, status, "id", "(none in file)", note))
    elif kind == "year":
        rows_hm.append((name, status, "constant", "(added)", note))
    elif kind == "single":
        k = "single (repaired)" if spec in FIXES and FIXES[spec][0] else "single"
        rows_hm.append((name, status, k, span_text(spec), note))
    elif kind == "list":
        rows_hm.append((name, status, "list ';'", span_text(spec), note))
    elif kind == "freetext":
        rows_hm.append((name, status, "free_text", span_text(spec), note))
    elif kind == "lang":
        rows_hm.append((name, status, "list ';' (cleaned)", span_text(spec), note))
        rows_hm.append((name + "_count", "proposed", "derived number", "(derived)", "number of distinct languages; 0 if 'None' ticked; blank if block unanswered or only an unusable write-in"))
        rows_hm.append(("ToolsHaveWorkedWith", "proposed", "list ';' (cleaned)", "(derived from tick-boxes and write-ins)", "frameworks/libraries/databases/platforms recognised in the language question; not counted as languages; empty = none recognised"))
    elif kind == "salary":
        rows_hm.append((name, status, "single (band text)", span_text(spec), note))
        rows_hm.append(("ConvertedCompYearly", "confirmed", "derived number", "(derived from the salary band)", "annual USD band midpoint; includes bonus; top band set by CONFIG"))
header_map = pd.DataFrame(rows_hm, columns=["canonical_name", "status", "kind", "original_header_or_span", "note"])
print(header_map.status.value_counts().to_dict())
print(header_map[["canonical_name", "status", "kind"]].to_string(index=False))

{'proposed': 61, 'confirmed': 8}
                canonical_name    status               kind
                    ResponseId  proposed                 id
                   survey_year  proposed           constant
                       Country confirmed             single
                       USState  proposed             single
                           Age confirmed             single
                        Gender  proposed             single
                     YearsCode confirmed  single (repaired)
                       DevType confirmed             single
                     CompTotal confirmed single (band text)
           ConvertedCompYearly confirmed     derived number
                      Industry  proposed             single
                   OrgDevCount  proposed  single (repaired)
                    RemoteWork  proposed             single
               RemoteEnjoyment  proposed             single
                RemoteLocation  proposed             single
      M

### Cell 7.2 - Assemble the canonical table
**Purpose:** put the columns together in the original questionnaire order, with the shared names.
**Decision made:** none new. Blanks stay blank; nothing is imputed, merged or encoded.

In [26]:
label_to_mid = {bnd[0]: bnd[3] for bnd in BANDS}
out = pd.DataFrame(index=df.index)
for kind, name, status, spec, expect, note in ITEMS:
    if kind == "id":
        out[name] = df.index.astype(int)
    elif kind == "year":
        out[name] = 2014
    elif kind in ("single", "freetext"):
        out[name] = df[RAW[spec]]
    elif kind == "list":
        out[name] = LISTS[name]
    elif kind == "lang":
        out[name] = language_list
        out[name + "_count"] = language_count
        out["ToolsHaveWorkedWith"] = tools_list
    elif kind == "salary":
        out[name] = df[RAW[spec]]
        out["ConvertedCompYearly"] = df[RAW[spec]].map(label_to_mid).astype(float)

assert list(out.columns) == list(header_map.canonical_name), "header map and table columns differ"
print("shape:", out.shape)
print(out.head(3).T.to_string(max_colwidth=60))

shape: (4623, 69)
                                                                                          1                                                   3                                                      6
ResponseId                                                                                1                                                   3                                                      6
survey_year                                                                            2014                                                2014                                                   2014
Country                                                                               India                                                Iran                                                Denmark
USState                                                                                 NaN                                                 NaN                                           

### Cell 7.3 - Validate
**Purpose:** prove the file is what it claims: right row count, no encoded columns, no empty strings, list columns well-formed, every single-answer column equal to the file (or to the file after the Gate 3 repair, for the repaired ones), missing values left as missing.
**Decision made:** none.

In [27]:
assert len(out) == len(df)
assert out["ResponseId"].is_unique
list_cols = [h for h, k in zip(header_map.canonical_name, header_map.kind) if k.startswith("list")]
for c in list_cols:
    s = out[c].dropna()
    assert not s.str.startswith(";").any() and not s.str.endswith(";").any() and not s.str.contains(";;").any(), c
    assert all(len(v.split(";")) == len(set(v.split(";"))) for v in s), "duplicate item in " + c
    print("%-28s | blank: %5d | mean items: %.2f" % (c, int(out[c].isna().sum()), s.str.split(";").str.len().mean()))
text_cols = [c for c in out.columns if not pd.api.types.is_numeric_dtype(out[c])]
for c in text_cols:
    assert not (out[c].dropna().str.strip() == "").any(), "empty string in " + c
numeric_cols = [c for c in out.columns if c not in text_cols]
na_like = {"None", "N/A", "NA", "n/a", "null", "NULL", "NaN", "nan"}
hits = {c: int(out[c].isin(na_like).sum()) for c in text_cols}
print("cells holding NA-like TEXT that is a real answer (read the CSV with keep_default_na=False):", {k: v for k, v in hits.items() if v})
print("numeric columns:", numeric_cols)

n_checked = 0
for kind, name, status, spec, expect, note in ITEMS:
    if kind in ("single", "freetext", "salary"):
        assert out[name].equals(df[RAW[spec]]), name + " differs from the repaired file column"
        if not (spec in FIXES and FIXES[spec][0]):
            assert out[name].equals(df_raw.loc[out.index, RAW[spec]]), name + " differs from the original file"
        n_checked += 1
print("single-answer and free-text columns checked against the file:", n_checked, "| repaired columns (compared with the repaired text):", sorted(repaired_cols))
print("missing per column:")
print(out.isna().sum()[out.isna().sum() > 0].to_string())
print("any one-hot style 0/1 columns:", [c for c in numeric_cols if set(out[c].dropna().unique()) <= {0, 1} and c != "survey_year"])

MobilePlatformsSupported     | blank:   514 | mean items: 2.10
PurchaseInvolvement          | blank:   239 | mean items: 1.29
PurchaseTypes                | blank:  1416 | mean items: 2.48
LanguageHaveWorkedWith       | blank:   313 | mean items: 3.11
ToolsHaveWorkedWith          | blank:  4033 | mean items: 1.08
TechExcitedAbout             | blank:  1252 | mean items: 2.40
TechProductsOwned            | blank:   523 | mean items: 2.61
StackOverflowUsage           | blank:   404 | mean items: 2.95
cells holding NA-like TEXT that is a real answer (read the CSV with keep_default_na=False): {'TimeSpend_NewFeatures': 202, 'TimeSpend_Refactoring': 326, 'TimeSpend_BugFixing': 180, 'TimeSpend_TechSupport': 1286, 'TimeSpend_Meetings': 389, 'TimeSpend_Learning': 272, 'TimeSpend_WebSurfing': 363, 'TimeSpend_LookingForJob': 3003, 'TimeSpend_StackExchange': 178, 'TechProductsOwned_OtherText': 4, 'AdvertisersRemembered': 81}
numeric columns: ['ResponseId', 'survey_year', 'ConvertedCompYearly', 'La

### Cell 7.4 - Save
**Purpose:** write `canonical_2014.csv` and `header_map_2014.csv`, then read the CSV back and compare.
**Decision made:** none.

In [28]:
out.to_csv("canonical_2014.csv", index=False)
header_map.to_csv("header_map_2014.csv", index=False)
back = pd.read_csv("canonical_2014.csv", keep_default_na=False, na_values=[""])
print("saved canonical_2014.csv | on disk:", back.shape, "| in memory:", out.shape)
assert back.shape == out.shape
assert list(back.columns) == list(out.columns)
assert (back.isna().values == out.isna().values).all(), "missing-value pattern changed on save"
for c in out.columns:
    if not pd.api.types.is_numeric_dtype(out[c]):
        assert (back[c].fillna("").values == out[c].fillna("").values).all(), "text changed on save in " + c
    else:
        assert np.allclose(back[c].fillna(-1).values, out[c].fillna(-1).values), "numbers changed on save in " + c
print("round trip identical: same shape, same missing pattern, same text and numbers in all", out.shape[1], "columns")
log("7.4 save", "canonical_2014.csv: " + str(out.shape[0]) + " rows x " + str(out.shape[1]) + " columns, not encoded")
print()
print("row log:")
for step, bf, af in row_log:
    print("  %-48s %5d -> %5d" % (step, bf, af))
print()
print("decision log:")
for step, text in decision_log:
    print(" ", step, "|", text)

saved canonical_2014.csv | on disk: (4623, 69) | in memory: (4623, 69)
round trip identical: same shape, same missing pattern, same text and numbers in all 69 columns
[LOG] 7.4 save - canonical_2014.csv: 4623 rows x 69 columns, not encoded

row log:
  1.6 option-label row moved out of data            7644 ->  7643
  4.2 junk rows (only Country/State answered)       7643 ->  7394
  4.4 salary answer blank                           7394 ->  6900
  4.4 'Rather not say'                              6900 ->  5770
  4.4 Student / Unemployed                          5770 ->  4623

decision log:
  1.1 encoding | file read with encoding=utf-8 (114 non-ASCII byte(s) in the file)
  1.2 load | loaded 7644 rows x 120 columns, all as text, default NA parsing off
  1.4 strip | 0 column name(s) had surrounding whitespace and were stripped
  1.5 blanks | 495389 empty/whitespace cells converted to NaN; 9568 literal 'None'-style cells kept as text
  1.6 option row | row 0 moved to option_labels (rows 764

## Written decision log (numbers come from this run)

The next cell prints the summary of what was kept, changed and dropped, built from the variables of this run, so the numbers cannot drift from the data.

In [29]:
print("OUTPUT: canonical_2014.csv,", out.shape[0], "rows x", out.shape[1], "columns; nothing one-hot encoded, imputed, merged or scaled.")
print("ROWS:   " + " -> ".join(str(r[2]) for r in row_log[:1]) + " | " + " | ".join("%s: %d -> %d" % r for r in row_log[1:]))
print("KEPT EXACTLY AS TYPED:", [n for n, k in zip(header_map.canonical_name, header_map.kind) if k in ("single", "single (band text)")])
print("REPAIRED (Excel damage / label fixes):", [n for n, k in zip(header_map.canonical_name, header_map.kind) if k == "single (repaired)"])
print("LISTS (';'-separated):", [n for n, k in zip(header_map.canonical_name, header_map.kind) if k.startswith("list")])
print("FREE TEXT (raw, drop before modelling):", [n for n, k in zip(header_map.canonical_name, header_map.kind) if k == "free_text"])
print("LANGUAGES:", len(all_names), "canonical names;", len(parsed), "write-ins parsed;", len(tool_names), "tool names; mean count %.2f" % language_count.mean())
print("HEADERS: ", header_map.status.value_counts().to_dict(), "(proposed names need checking against your Canonical Mapping sheet)")
print("LOAD WITH: pd.read_csv(path, keep_default_na=False, na_values=[''])  -> real answers such as 'None' survive:", {k: v for k, v in hits.items() if v})
print("LEFT FOR THE POOLED PIPELINE (fit on training years only): one-hot / multi-hot, rare-level merging, imputation, scaling, feature selection")

OUTPUT: canonical_2014.csv, 4623 rows x 69 columns; nothing one-hot encoded, imputed, merged or scaled.
ROWS:   7643 | 4.2 junk rows (only Country/State answered): 7643 -> 7394 | 4.4 salary answer blank: 7394 -> 6900 | 4.4 'Rather not say': 6900 -> 5770 | 4.4 Student / Unemployed: 5770 -> 4623
KEPT EXACTLY AS TYPED: ['Country', 'USState', 'Age', 'Gender', 'DevType', 'CompTotal', 'Industry', 'RemoteWork', 'RemoteEnjoyment', 'RemoteLocation', 'TimeSpend_NewFeatures', 'TimeSpend_Refactoring', 'TimeSpend_BugFixing', 'TimeSpend_TechSupport', 'TimeSpend_Meetings', 'TimeSpend_Learning', 'TimeSpend_WebSurfing', 'TimeSpend_LookingForJob', 'TimeSpend_StackExchange', 'BudgetOutsideExpenditure', 'OpSys', 'ChangedJobsLast12m', 'JobFoundVia', 'OpenToNewJob', 'RecruiterContactFrequency', 'ContactPref_Email', 'ContactPref_LinkedIn', 'ContactPref_Phone', 'ContactPref_CareersMessage', 'ContactPref_Twitter', 'RecruitEmail_Personalized', 'RecruitEmail_MentionsCode', 'RecruitEmail_Salary', 'RecruitEmail_Te